In [1]:
import os
import shutil
import yaml

In [ ]:
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/csdd.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/hyp.simulation.yaml
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/0389.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/1356.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/0534.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/0333.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/1038.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/0069.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/2013.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/1586.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/0354.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/1921.txt
/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws/labels/val/1651.

In [3]:
SOURCE = '/kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws' 
DEST = '/kaggle/working/ws'

if os.path.exists(SOURCE):
    if os.path.exists(DEST):
        shutil.rmtree(DEST)
    
    shutil.copytree(SOURCE, DEST)
    print(f" Data copied from {SOURCE} to {DEST}")
else:
    print(" Source folder not found.")

 Data copied from /kaggle/input/datasets/arnavde/csdd-ws1x-modified/CSDD_ws to /kaggle/working/ws


In [4]:
yaml_path = '/kaggle/working/ws/csdd.yaml'
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

config['path'] = '/kaggle/working/ws' 
config['train'] = 'img/train'
config['val'] = 'img/val'
config['test'] = 'img/test'

with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(" csdd.yaml paths updated to /kaggle/working/ws/")

 csdd.yaml paths updated to /kaggle/working/ws/


In [5]:
if not os.path.exists('yolov5'):
    print("Cloning YOLOv5 repository...")
    !git clone https://github.com/ultralytics/yolov5
    %cd yolov5
    print("Installing dependencies...")
    !pip install -r requirements.txt -q
    !pip install albumentations==1.4.11 --force-reinstall # to use the blur and other albumentations so that image detection works better in the hazy camera feed of gazebo
else:
    %cd yolov5
    print("YOLOv5 environment already initialized.")

print("Environment Ready: YOLOv5 and all dependencies are active.")

Cloning YOLOv5 repository...
Cloning into 'yolov5'...
remote: Enumerating objects: 17889, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 17889 (delta 33), reused 9 (delta 9), pack-reused 17850 (from 5)
Receiving objects: 100% (17889/17889), 17.06 MiB | 24.12 MiB/s, done.
Resolving deltas: 100% (12187/12187), done.
/kaggle/working/yolov5
Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/

In [6]:
if os.path.exists('/kaggle/working/ws/img'):
    os.rename('/kaggle/working/ws/img', '/kaggle/working/ws/images')
    print(" Folder 'img' renamed to 'images'")

yaml_path = '/kaggle/working/ws/csdd.yaml'
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

config['train'] = 'images/train'
config['val'] = 'images/val'
config['test'] = 'images/test'

with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(" csdd.yaml updated with 'images' paths.")

 Folder 'img' renamed to 'images'
 csdd.yaml updated with 'images' paths.


In [7]:
source = '/kaggle/working/ws/csdd.yaml'
destination = '/kaggle/working/yolov5/data'

os.makedirs(os.path.dirname(destination), exist_ok=True)

try:
    shutil.move(source, destination)
    print("File moved successfully!")
except FileNotFoundError:
    print("Error: The source file was not found.")
except PermissionError:
    print("Error: Permission denied.")

File moved successfully!


In [8]:
source = '/kaggle/working/ws/hyp.simulation.yaml'
destination = '/kaggle/working/yolov5/data/hyps'

os.makedirs(os.path.dirname(destination), exist_ok=True)

try:
    shutil.move(source, destination)
    print("File moved successfully!")
except FileNotFoundError:
    print("Error: The source file was not found.")
except PermissionError:
    print("Error: Permission denied.")

File moved successfully!


In [9]:
%cd /kaggle/working/yolov5

%env PYTHONWARNINGS=ignore

!python train.py \
                 --img 1024 \
                 --batch 32 \
                 --epochs 10 \
                 --data data/csdd.yaml \
                 --weights yolov5s.pt \
                 --device 0 \
                 --optimizer SGD \
                 --hyp data/hyps/hyp.simulation.yaml \
                 --evolve 5 \
                 --name CSDD_Evaluation_Model \
                 --exist-ok | grep -E "Epoch|mAP|Class" # Ensures output of 1 line per epoch to prevent flooding of output

/kaggle/working/yolov5
env: PYTHONWARNINGS=ignore
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-04-19 04:24:17.836847: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776572658.058599      93 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776572658.126862      93 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776572658.636381      93 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776572658.636477      93 computation_p

In [10]:
%cd /kaggle/working

!zip -r YOLOv5_Full_Evolution.zip yolov5 -q

/kaggle/working
